## LSTM (Long Short Term Memory):- 
Long Short term memory is a type of the sequential model, that takes one timestep at a time in order process it and produced two state, cell state and the hidden state.

Cell staate c(t) is the long term memory, that contains the data which are important for long term. It contains the understanding that the model has understood so far that is important in future timestep and they might depend upon but we dont know which and what will be important.

Hidden state h(t) is the short term memory, that contains the contextual understanding/summary or the snapshot of the lstm model throughout the process till the previous timestep. It does hold the information that is important and crucial for now.

It makes the LSTM, different from the RNN because, RNN stores everything in one state due to which the information that we will be loosing is the long term information, which removes the long term dependency/quality information and it causes the lack of prediciton quality.

Whereas, in LSTM there are two state, cell state only holds the information that are found in early timestep as well as the information that can be crucial for future timesteps. And the short term memory s like the hidden state of rnn, as it holds the contextual understanding till the previous timestep. And the short term memory contains the information crucial in current scenario. And this provides us the ability to play with the LTM and STM like, we can decide what should we forget or is no longer important, what should we consider the information that can be important in future and what are the information we neet to add to STM from LTM which can be crucial for next time step.

Provides us control to what to forget, what to remember and what to consider right now.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

In [2]:
nltk.download("punkt_tab")
nltk.download("punkt")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Nitro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### we cannot load all the text from the text.txt file all at once because of the RAM gettig full, so we will use generator to produce desired number of text at a time

In [3]:
def file():
    with open("smaller.txt", "r") as f:
            for line in f:
                yield line

In [74]:
def tokens_generator(f):
    tokens = []
    try:
        while True:
            text = next(f)
            tokens.extend(word_tokenize(text.lower()))
    except StopIteration:
        return tokens

In [5]:
def voccabulary():
    f = file()
    tokens = tokens_generator(f)
    voccab = {
        '<UNK>' : 0,
        '<PAD>' : 1,
    }
    for token in Counter(tokens):
        voccab[token] = len(voccab)
    return voccab

In [6]:
voccab = voccabulary()

In [7]:
voccab

{'<UNK>': 0,
 '<PAD>': 1,
 'usually': 2,
 'he': 3,
 'would': 4,
 'be': 5,
 'tearing': 6,
 'around': 7,
 'the': 8,
 'living': 9,
 'room': 10,
 'playing': 11,
 'with': 12,
 'his': 13,
 'toys': 14,
 'but': 15,
 'just': 16,
 'one': 17,
 'look': 18,
 'at': 19,
 'a': 20,
 'minion': 21,
 'sent': 22,
 'him': 23,
 'practically': 24,
 'catatonic': 25,
 'that': 26,
 'had': 27,
 'been': 28,
 'megan': 29,
 's': 30,
 'plan': 31,
 'when': 32,
 'she': 33,
 'got': 34,
 'dressed': 35,
 'earlier': 36,
 'd': 37,
 'seen': 38,
 'movie': 39,
 'almost': 40,
 'by': 41,
 'mistake': 42,
 'considering': 43,
 'was': 44,
 'little': 45,
 'young': 46,
 'for': 47,
 'pg': 48,
 'cartoon': 49,
 'older': 50,
 'cousins': 51,
 'along': 52,
 'her': 53,
 'brothers': 54,
 'mason': 55,
 'often': 56,
 'exposed': 57,
 'to': 58,
 'things': 59,
 'were': 60,
 'liked': 61,
 'think': 62,
 'being': 63,
 'surrounded': 64,
 'adults': 65,
 'and': 66,
 'kids': 67,
 'reason': 68,
 'why': 69,
 'such': 70,
 'good': 71,
 'talker': 72,
 'age': 

In [8]:
len(voccab)

25629

In [123]:
def text_to_indices(voccab):
    f = file()
    dataset = []
    max_len = 0
    # extract text from file
    try:
        while True:
            # tokenize
            indices = []
            text = next(f)
            tokens = word_tokenize(text.lower())
            if max_len < len(tokens):
                max_len = len(tokens)
            # replace with respective integer from voccab
            for token in tokens:
                indices.append(voccab[token])
            # preparing the dataset from indices
            print(indices)
            for element in range(len(indices) - 1):
                dataset.append(indices[ : element + 2])
            print(dataset)
            break
    except StopIteration:
        return  max_len, dataset

In [124]:
indices = text_to_indices(voccab)

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[[2, 3], [2, 3, 4], [2, 3, 4, 5], [2, 3, 4, 5, 6], [2, 3, 4, 5, 6, 7], [2, 3, 4, 5, 6, 7, 8], [2, 3, 4, 5, 6, 7, 8, 9], [2, 3, 4, 5, 6, 7, 8, 9, 10], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]


In [110]:
indices

### we need to create the dataset (inputs) for the LSTM model
As we are trying to make the next word predction so, in a sequence there can be different size of sentence so we need to perfrom the padding to make them of the same size as well.

In [106]:
def dataset_preparatioin(max_length, indices):
    dataset = []
    for i in indices:
        print(i)
        dataset.append([1]*(max_length - len(i)) + i)
        print(len([1]*(max_length - len(i)) + i))
        break
    print(dataset)

In [107]:
dataset_preparatioin(indices[0], indices[1])

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
93
[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]


In [72]:
a = [1,2,3,4,5,6,7,8,9,1,2,4,7,5,3]
len(a)

15

In [115]:
for i in range(len(a) - 1):
    print(a[:i+2])

[1, 2]
[1, 2, 3]
[1, 2, 3, 4]
[1, 2, 3, 4, 5]
[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7, 8]
[1, 2, 3, 4, 5, 6, 7, 8, 9]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 4]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 4, 7]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 4, 7, 5]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 4, 7, 5, 3]
